# Example: Fine-tune BERT on a custom dataset

In [7]:
from datasets import load_dataset
from transformers import (
    TrainingArguments, Trainer, 
    BertTokenizer, BertForSequenceClassification)
import torch
import numpy as np
import evaluate

In [8]:
# Load model and tokenizer
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
# Load dataset (IMDb movie reviews) 
dataset = load_dataset("imdb")

In [15]:
# Tokenize the dataset

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

tokenized_ds = dataset.map(tokenize, batched=True)
tokenized_ds = tokenized_ds.rename_column("label", "labels")
tokenized_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [16]:
# Define metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

In [17]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./bert_output",
    eval_strategy="epoch",              # evaluated every epoch
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=50,                   # print progress every 50 steps
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",                   # disable WandB unless you use it
)

In [18]:
# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"].shuffle(seed=42).select(range(2000)),  # small subset for speed
    eval_dataset=tokenized_ds["test"].select(range(500)),
    compute_metrics=compute_metrics,
)

In [19]:
# Train & evaluate
trainer.train()
metrics = trainer.evaluate()
print(metrics)

Epoch,Training Loss,Validation Loss


Visualize logs in TensorBoard

    tensorboard --logdir ./logs

Open http://localhost:6006
 and you’ll see loss and metric plots per epoch.